In [2]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Any
from pathlib import Path

# =========================
# CONFIG
# =========================

INPUT_FILE = "kids_english_dataset_10000_FINAL.json"
OUTPUT_FILE = "kids_english_dataset_10000_retopic.json"
REPORT_FILE = "kids_english_dataset_10000_topic_report.json"

# =========================
# NORMALIZATION
# =========================

def normalize_word(word: str) -> str:
    if not isinstance(word, str):
        return ""
    word = word.strip().lower()
    word = re.sub(r"\s+", " ", word)
    return word

# =========================
# PRODUCTION TAXONOMY
# =========================
#
# Main idea:
# - topic: user-facing category
# - sub_topic: parent bucket
# - pos_hint: rough grammatical hint for internal use
#
# You can freely rename these labels to fit your app UI.

CATEGORY_RULES = {
    # Grammar / Function words
    "articles_determiners": {
        "topic": "Articles & Determiners",
        "sub_topic": "Grammar",
        "pos_hint": "DET",
        "words": {
            "the", "a", "an", "this", "that", "these", "those",
            "some", "any", "all", "each", "every", "another",
            "other", "one", "many", "much", "few", "little",
        },
    },
    "pronouns": {
        "topic": "Pronouns",
        "sub_topic": "Grammar",
        "pos_hint": "PRON",
        "words": {
            "i", "you", "he", "she", "it", "we", "they",
            "me", "him", "her", "us", "them",
            "my", "your", "his", "its", "our", "their",
            "mine", "yours", "hers", "ours", "theirs",
            "myself", "yourself", "himself", "herself", "itself",
            "ourselves", "themselves",
        },
    },
    "question_words": {
        "topic": "Question Words",
        "sub_topic": "Grammar",
        "pos_hint": "WH",
        "words": {"what", "who", "whom", "whose", "when", "where", "why", "how", "which"},
    },
    "prepositions": {
        "topic": "Prepositions",
        "sub_topic": "Grammar",
        "pos_hint": "ADP",
        "words": {
            "in", "on", "at", "to", "from", "with", "for", "about", "of",
            "by", "under", "over", "between", "into", "through", "after",
            "before", "around", "near", "behind", "beside", "inside",
            "outside", "out",
        },
    },
    "conjunctions": {
        "topic": "Conjunctions",
        "sub_topic": "Grammar",
        "pos_hint": "CCONJ",
        "words": {"and", "but", "or", "so", "yet", "because", "although", "if", "while", "than"},
    },
    "auxiliary_verbs": {
        "topic": "Auxiliary & Modal Verbs",
        "sub_topic": "Grammar",
        "pos_hint": "AUX",
        "words": {
            "am", "is", "are", "was", "were", "be", "been", "being",
            "do", "does", "did",
            "have", "has", "had",
            "can", "could", "may", "might", "must", "shall", "should",
            "will", "would",
        },
    },
    "negation_quantifiers": {
        "topic": "Negation & Quantity",
        "sub_topic": "Grammar",
        "pos_hint": "ADV/DET",
        "words": {"not", "no", "none", "more", "most", "less", "least", "only", "enough"},
    },

    # Semantic / everyday usage
    "home_places": {
        "topic": "Home & Places",
        "sub_topic": "Everyday Life",
        "pos_hint": "NOUN",
        "words": {
            "home", "house", "room", "kitchen", "bedroom", "school", "park",
            "shop", "store", "city", "town", "village", "site", "place",
            "page", "office", "classroom",
        },
    },
    "time_words": {
        "topic": "Time",
        "sub_topic": "Everyday Life",
        "pos_hint": "NOUN/ADV",
        "words": {
            "time", "today", "tomorrow", "yesterday", "morning", "afternoon",
            "evening", "night", "week", "month", "year", "day", "hour",
            "minute", "second", "now", "later", "soon",
        },
    },
    "technology_internet": {
        "topic": "Technology & Internet",
        "sub_topic": "Modern Life",
        "pos_hint": "NOUN/VERB",
        "words": {
            "web", "website", "internet", "online", "search", "click", "page",
            "site", "email", "computer", "phone", "tablet", "app", "contact",
            "login", "download", "upload",
        },
    },
    "media_information": {
        "topic": "Media & Information",
        "sub_topic": "Modern Life",
        "pos_hint": "NOUN",
        "words": {"news", "story", "book", "video", "music", "radio", "tv", "article"},
    },
    "business_money": {
        "topic": "Business & Money",
        "sub_topic": "Modern Life",
        "pos_hint": "NOUN",
        "words": {"business", "money", "buy", "sell", "price", "shop", "market", "cost", "free"},
    },
    "actions_common": {
        "topic": "Common Actions",
        "sub_topic": "Everyday Life",
        "pos_hint": "VERB",
        "words": {
            "see", "look", "go", "come", "run", "walk", "eat", "drink",
            "play", "read", "write", "speak", "talk", "listen", "use",
            "make", "take", "give", "find", "help", "open", "close",
            "search", "contact",
        },
    },
    "people_social": {
        "topic": "People & Social",
        "sub_topic": "Everyday Life",
        "pos_hint": "NOUN/PRON",
        "words": {"friend", "family", "teacher", "doctor", "baby", "child", "children", "parent"},
    },
}

# Flatten exact-word lookup
EXACT_WORD_MAP: Dict[str, Tuple[str, str, str]] = {}
for rule in CATEGORY_RULES.values():
    for w in rule["words"]:
        EXACT_WORD_MAP[w] = (rule["topic"], rule["sub_topic"], rule["pos_hint"])

# =========================
# HEURISTICS / FALLBACKS
# =========================

NUMBER_WORDS = {
    "zero", "one", "two", "three", "four", "five", "six",
    "seven", "eight", "nine", "ten", "first", "second", "third"
}

COMMON_ADVERBS = {
    "here", "there", "very", "too", "also", "again", "together",
    "away", "back", "almost", "already"
}

COMMON_ADJECTIVES = {
    "new", "old", "big", "small", "good", "bad", "happy", "sad",
    "fast", "slow", "hot", "cold", "free"
}

COMMON_VERB_SUFFIXES = ("ing", "ed", "en")
COMMON_ADVERB_SUFFIXES = ("ly",)
COMMON_NOUN_SUFFIXES = ("tion", "sion", "ment", "ness", "ity", "ship", "er", "or")

def classify_by_heuristic(word: str, vietnamese: str = "", example: str = "") -> Tuple[str, str, str, str]:
    """
    Returns:
        topic, sub_topic, pos_hint, method
    """
    if not word:
        return "Uncategorized", "Unknown", "UNK", "empty"

    # Exact match first
    if word in EXACT_WORD_MAP:
        topic, sub_topic, pos_hint = EXACT_WORD_MAP[word]
        return topic, sub_topic, pos_hint, "exact"

    # Numbers
    if word in NUMBER_WORDS or word.isdigit():
        return "Numbers", "Math & Logic", "NUM", "heuristic"

    # Common adverbs
    if word in COMMON_ADVERBS:
        return "Adverbs", "Grammar", "ADV", "heuristic"

    # Common adjectives
    if word in COMMON_ADJECTIVES:
        return "Describing Words", "Grammar", "ADJ", "heuristic"

    # Morphology hints
    if word.endswith(COMMON_ADVERB_SUFFIXES):
        return "Adverbs", "Grammar", "ADV", "suffix"

    if word.endswith(COMMON_VERB_SUFFIXES):
        return "Common Actions", "Everyday Life", "VERB", "suffix"

    if word.endswith(COMMON_NOUN_SUFFIXES):
        return "Objects & Ideas", "Vocabulary", "NOUN", "suffix"

    # Vietnamese hint fallback
    vn = normalize_word(vietnamese)
    if vn:
        if vn in {"và", "nhưng", "hoặc"}:
            return "Conjunctions", "Grammar", "CCONJ", "vietnamese_hint"
        if vn in {"với", "từ", "về", "trong", "trên", "dưới"}:
            return "Prepositions", "Grammar", "ADP", "vietnamese_hint"
        if vn in {"ai", "gì", "khi", "đâu", "tại sao", "như thế nào"}:
            return "Question Words", "Grammar", "WH", "vietnamese_hint"
        if vn in {"không", "chưa"}:
            return "Negation & Quantity", "Grammar", "ADV/DET", "vietnamese_hint"
        if vn in {"mới", "lớn", "nhỏ", "đẹp", "xấu"}:
            return "Describing Words", "Grammar", "ADJ", "vietnamese_hint"
        if vn in {"nhìn thấy", "sử dụng", "tìm kiếm", "liên hệ"}:
            return "Common Actions", "Everyday Life", "VERB", "vietnamese_hint"
        if vn in {"thời gian", "hôm nay", "ngày", "tuần", "tháng"}:
            return "Time", "Everyday Life", "NOUN/ADV", "vietnamese_hint"
        if vn in {"tin tức"}:
            return "Media & Information", "Modern Life", "NOUN", "vietnamese_hint"
        if vn in {"địa điểm", "trang", "trang chủ", "nhà"}:
            return "Home & Places", "Everyday Life", "NOUN", "vietnamese_hint"

    # Example hint
    ex = normalize_word(example)
    if ex:
        if any(token in ex for token in ["website", "web", "internet", "search"]):
            return "Technology & Internet", "Modern Life", "NOUN/VERB", "example_hint"
        if any(token in ex for token in ["home", "school", "park", "room"]):
            return "Home & Places", "Everyday Life", "NOUN", "example_hint"

    # Final fallback
    return "General Vocabulary", "Vocabulary", "UNK", "fallback"

# =========================
# MAIN PIPELINE
# =========================

def classify_record(item: Dict[str, Any]) -> Dict[str, Any]:
    word = normalize_word(item.get("word", ""))
    vietnamese = item.get("vietnamese", "")
    example = item.get("example", "")

    topic, sub_topic, pos_hint, method = classify_by_heuristic(word, vietnamese, example)

    updated = dict(item)
    updated["topic"] = topic
    updated["sub_topic"] = sub_topic
    updated["pos_hint"] = pos_hint
    updated["topic_assignment_method"] = method
    return updated

def generate_report(data: List[Dict[str, Any]]) -> Dict[str, Any]:
    topic_counter = Counter()
    sub_topic_counter = Counter()
    pos_counter = Counter()
    method_counter = Counter()
    topic_examples = defaultdict(list)

    for item in data:
        topic = item.get("topic", "Unknown")
        sub_topic = item.get("sub_topic", "Unknown")
        pos = item.get("pos_hint", "UNK")
        method = item.get("topic_assignment_method", "unknown")
        word = item.get("word", "")

        topic_counter[topic] += 1
        sub_topic_counter[sub_topic] += 1
        pos_counter[pos] += 1
        method_counter[method] += 1

        if len(topic_examples[topic]) < 10:
            topic_examples[topic].append(word)

    return {
        "total_records": len(data),
        "topics": dict(topic_counter.most_common()),
        "sub_topics": dict(sub_topic_counter.most_common()),
        "pos_hints": dict(pos_counter.most_common()),
        "assignment_methods": dict(method_counter.most_common()),
        "topic_examples": dict(topic_examples),
    }

def reassign_topics(input_file: str, output_file: str, report_file: str) -> None:
    input_path = Path(input_file)
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_file}")

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Expected top-level JSON array (list of word records).")

    updated_data = []
    for idx, item in enumerate(data):
        if not isinstance(item, dict):
            # Skip malformed records but keep track
            updated_data.append({
                "original_index": idx,
                "topic": "Invalid Record",
                "sub_topic": "System",
                "pos_hint": "UNK",
                "topic_assignment_method": "invalid_record",
                "raw_value": item,
            })
            continue

        updated_data.append(classify_record(item))

    report = generate_report(updated_data)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(updated_data, f, ensure_ascii=False, indent=2)

    with open(report_file, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"Done.")
    print(f"Output file : {output_file}")
    print(f"Report file : {report_file}")
    print(f"Total items : {report['total_records']}")
    print("Top topics  :")
    for topic, count in list(report["topics"].items())[:10]:
        print(f"  - {topic}: {count}")

# =========================
# RUN
# =========================

if __name__ == "__main__":
    reassign_topics(INPUT_FILE, OUTPUT_FILE, REPORT_FILE)

print("Output path:", Path(OUTPUT_FILE).resolve())
print("Report path:", Path(REPORT_FILE).resolve())

Done.
Output file : kids_english_dataset_10000_retopic.json
Report file : kids_english_dataset_10000_topic_report.json
Total items : 8900
Top topics  :
  - General Vocabulary: 6635
  - Common Actions: 1271
  - Objects & Ideas: 590
  - Adverbs: 167
  - Home & Places: 31
  - Prepositions: 26
  - Pronouns: 21
  - Technology & Internet: 20
  - Time: 20
  - Auxiliary & Modal Verbs: 19
Output path: /content/kids_english_dataset_10000_retopic.json
Report path: /content/kids_english_dataset_10000_topic_report.json
